# XAI-project — Interpretable Prediction of Mega-Fires in Chile

**Course:** INF-473 Explainable AI · UTFSM · Prof. Raquel Pezoa Rivera  
**Authors:** Eduardo Morales · Octavia Jara · Benjamín Reyes

**Notebook:** L1 mega-fire baseline — XGBoost + Tree SHAP (stratified 5-fold CV). Trains in seconds; writes figures to `latex/images/` and the model to `data/models/`.

**How to run:** requires `data/processed/conaf_enriched_2016_2017.parquet` (or extract `conaf_enriched_latest.tar.gz`).

> See the repository [`README.md`](../README.md) for setup, the data pipeline, and reproduction steps.

# XGBoost baseline — L1 mega-fire classifier (preliminary)

Binary classification of post-ignition fire events into *mega-fire* (L1: burned area
$\geq 1000$ ha) vs ordinary fire, over the 2016-2017 modelable subset (four study
regions). Only **ex-ante** predictors are used. Evaluation is 5-fold stratified
(ROC-AUC / PR-AUC); Tree SHAP is computed on a final model retrained on the full subset.

Preliminary E2 results: the positive class is extremely rare (78 events), so metrics
have high variance and are reported as such.

In [1]:
import json, time, platform, sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import xgboost as xgb
import shap
import sklearn
from sklearn.metrics import (average_precision_score, roc_auc_score,
                             classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, precision_recall_curve)
from sklearn.model_selection import StratifiedKFold, cross_val_predict

warnings.filterwarnings("ignore", category=FutureWarning)

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.config import BASE_DIR, DATA_PROCESSED, DATA_MODELS
from src.modeling_features import FEATURE_COLS, MEGAFIRE_HA_THRESHOLD, RANDOM_STATE, N_SPLITS, XGB_PARAMS

FIG_DIR = BASE_DIR / "latex" / "images"; FIG_DIR.mkdir(parents=True, exist_ok=True)

FOCUS_REGIONS = ["Maule", "Biobío", "Araucanía", "O'Higgins"]
TARGET_COL = "superficie_quemada_total_ha"

## 1. Load dataset, restrict to the four study regions, drop out-of-coverage

The modelable subset is the 8,664 events in the four study regions; events whose ERA5-Land
match falls outside the extraction box (`out_of_coverage`) are dropped before training,
leaving 8,650 events with valid coverage.

In [2]:
df = pd.read_parquet(DATA_PROCESSED / "conaf_enriched_2012_2018.parquet")
print(f"national 2012-2018: {len(df):,} rows x {df.shape[1]} cols")

df = df[df["region"].isin(FOCUS_REGIONS)].copy()
print(f"modelable subset (4 regions): {len(df):,}")
print(df["region"].value_counts().to_string())

df = df[df["era5_match_quality"] != "out_of_coverage"].copy()
print(f"with valid ERA5 coverage (training set): {len(df):,}")

national 2012-2018: 42,963 rows x 71 cols
modelable subset (4 regions): 31,118
region
Biobío       17717
Araucanía     7090
Maule         4703
O'Higgins     1608
with valid ERA5 coverage (training set): 31,057


## 2. Target definition (L1)

In [3]:
df = df.dropna(subset=[TARGET_COL]).copy()
df["is_megafire"] = (df[TARGET_COL] >= MEGAFIRE_HA_THRESHOLD).astype(int)

n_pos = int(df["is_megafire"].sum())
n_neg = len(df) - n_pos
spw_global = n_neg / n_pos
print(f"L1 threshold: >= {MEGAFIRE_HA_THRESHOLD} ha")
print(f"positives: {n_pos} ({n_pos/len(df):.2%})   negatives: {n_neg:,}")
print(f"scale_pos_weight (global): {spw_global:.1f}")

L1 threshold: >= 1000 ha
positives: 78 (0.25%)   negatives: 30,979
scale_pos_weight (global): 397.2


## 3. Ex-ante feature set

Only predictors available *at ignition time* are kept, as an explicit whitelist matching
the paper's consolidated training feature table. Post-ignition outcomes are **excluded**:
the per-vegetation burned-area breakdown (`superficie_quemada_*_ha`, whose sum is the
target) and the fire `duracion_minutos` would leak the label; so do the operational
labels (`escenario`, `alerta`, `causa`), the MODIS/FIRMS audit columns and the ERA5 join
diagnostics. The whitelist below is the only source of predictors.

In [4]:
ts = pd.to_datetime(df["fecha_hora_inicio"], errors="coerce")
df["month"] = ts.dt.month
df["hour"] = ts.dt.hour
df["day_of_year"] = ts.dt.dayofyear

# Ex-ante predictor whitelist == paper training feature table (fuente única: src/modeling_features.py).
feature_cols = list(FEATURE_COLS)

CAT_COLS = ["region", "provincia", "comuna"]
for c in CAT_COLS:
    df[c] = pd.Categorical(df[c]).codes   # ordinal codes; -1 for NaN

missing = [c for c in feature_cols if c not in df.columns]
assert not missing, f"missing expected ex-ante columns: {missing}"
LEAK = ["superficie_quemada", "duracion_minutos", "escenario", "fli_estimado",
        "modis_", "label_l2", "alerta", "causa", "era5_dist", "era5_dt", "era5_match"]
leaked = [c for c in feature_cols if any(t in c for t in LEAK)]
assert not leaked, f"leakage columns present: {leaked}"

X = df[feature_cols]
y = df["is_megafire"]
print(f"ex-ante features: {len(feature_cols)}")
print(feature_cols)

ex-ante features: 44
['latitud', 'longitud', 'region', 'provincia', 'comuna', 'month', 'hour', 'day_of_year', 't2m', 'd2m', 'u10', 'v10', 'tp', 'ssrd', 'stl1', 'stl2', 'stl3', 'stl4', 'swvl1', 'swvl2', 'swvl3', 'swvl4', 'pev', 'e', 'evavt', 'lai_hv', 'lai_lv', 'slt', 'lsm', 'cvh', 'cvl', 'tvh', 'tvl', 't2m_celsius', 'd2m_celsius', 'stl1_celsius', 'stl2_celsius', 'stl3_celsius', 'stl4_celsius', 'relative_humidity', 'vpd_hpa', 'wind_speed', 'wind_direction', 'tp_mm']


## 4. Cross-validation (5-fold stratified)

With 78 positives a single split is unstable, so we report mean ± std of ROC-AUC and
PR-AUC across folds. `scale_pos_weight` is recomputed per fold from the training split.

In [5]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
roc_scores, pr_scores = [], []
for k, (tr, va) in enumerate(skf.split(X, y), 1):
    spw = (y.iloc[tr] == 0).sum() / max((y.iloc[tr] == 1).sum(), 1)
    m = xgb.XGBClassifier(**XGB_PARAMS, scale_pos_weight=spw)
    m.fit(X.iloc[tr], y.iloc[tr])
    p = m.predict_proba(X.iloc[va])[:, 1]
    roc = roc_auc_score(y.iloc[va], p)
    pr = average_precision_score(y.iloc[va], p)
    roc_scores.append(roc); pr_scores.append(pr)
    print(f"fold {k}: ROC-AUC={roc:.3f}  PR-AUC={pr:.3f}  (val positives={int((y.iloc[va]==1).sum())})")

roc_scores = np.array(roc_scores); pr_scores = np.array(pr_scores)
print()
print(f"ROC-AUC: {roc_scores.mean():.3f} +/- {roc_scores.std():.3f}")
print(f"PR-AUC : {pr_scores.mean():.3f} +/- {pr_scores.std():.3f}")
print(f"PR-AUC baseline (prevalence): {y.mean():.4f}")

fold 1: ROC-AUC=0.918  PR-AUC=0.189  (val positives=16)
fold 2: ROC-AUC=0.969  PR-AUC=0.093  (val positives=16)
fold 3: ROC-AUC=0.894  PR-AUC=0.106  (val positives=15)
fold 4: ROC-AUC=0.934  PR-AUC=0.275  (val positives=15)
fold 5: ROC-AUC=0.869  PR-AUC=0.068  (val positives=16)

ROC-AUC: 0.917 +/- 0.034
PR-AUC : 0.146 +/- 0.076
PR-AUC baseline (prevalence): 0.0025


### Out-of-fold confusion matrix and report (threshold 0.5)

In [6]:
oof_clf = xgb.XGBClassifier(**XGB_PARAMS, scale_pos_weight=spw_global)
y_oof = cross_val_predict(oof_clf, X, y, cv=skf, method="predict")
print(classification_report(y, y_oof, target_names=["fire", "megafire"], digits=3))

fig, ax = plt.subplots(figsize=(4, 4))
ConfusionMatrixDisplay(confusion_matrix(y, y_oof),
                       display_labels=["fire", "megafire"]).plot(ax=ax, colorbar=False)
ax.set_title("Out-of-fold confusion matrix (L1)")
fig.tight_layout(); fig.savefig(FIG_DIR / "cm_l1.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("saved", FIG_DIR / "cm_l1.png")

              precision    recall  f1-score   support

        fire      0.998     0.996     0.997     30979
    megafire      0.148     0.256     0.188        78

    accuracy                          0.994     31057
   macro avg      0.573     0.626     0.592     31057
weighted avg      0.996     0.994     0.995     31057

saved /home/imorales/Desktop/XAI/XAI-project/latex/images/cm_l1.png


### Out-of-fold precision-recall curve

The full precision/recall trade-off behind the single PR-AUC number, swept over every
threshold on the out-of-fold probabilities. The dashed line is the random baseline
(= prevalence); the red dot marks the default 0.5 threshold used in the confusion matrix
above. With 78 positives the curve is necessarily stepped.

In [ ]:
# Same out-of-fold protocol as the confusion matrix above, but keeping the
# probabilities so we can sweep every threshold instead of fixing it at 0.5.
oof_scores = cross_val_predict(oof_clf, X, y, cv=skf, method="predict_proba")[:, 1]
prec, rec, thr = precision_recall_curve(y, oof_scores)
ap = average_precision_score(y, oof_scores)
prevalence = y.mean()

fig, ax = plt.subplots(figsize=(4.5, 4))
ax.step(rec, prec, where="post", color="#2c7fb8", lw=1.8,
        label=f"PR curve (PR-AUC = {ap:.3f})")
ax.axhline(prevalence, ls="--", color="gray",
           label=f"baseline = prevalence ({prevalence:.4f})")

# Operating point of the default 0.5 threshold (matches the confusion matrix above).
j = min(int(np.searchsorted(thr, 0.5)), len(thr) - 1)
ax.scatter(rec[j], prec[j], color="#c0392b", zorder=5,
           label=f"threshold 0.5  (R={rec[j]:.2f}, P={prec[j]:.2f})")

ax.set_xlabel("recall"); ax.set_ylabel("precision")
ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
ax.set_title("Out-of-fold precision-recall curve (L1)")
ax.legend(loc="upper right", fontsize=8, framealpha=0.9)
print(f"PR-AUC (OOF) = {ap:.3f}   baseline (prevalence) = {prevalence:.4f}")
fig  # render inline (Jupyter displays the Figure object even with the Agg backend)

## 5. Final model on the full subset (for SHAP)

In [8]:
t0 = time.perf_counter()
model = xgb.XGBClassifier(**XGB_PARAMS, scale_pos_weight=spw_global)
model.fit(X, y)
train_seconds = time.perf_counter() - t0
print(f"final model trained on {len(X):,} events in {train_seconds:.2f}s")

final model trained on 31,057 events in 1.54s


## 6. Tree SHAP explanations

Global drivers (mean |SHAP| and beeswarm), a dependence plot for the top feature, and a
local waterfall for the largest mega-fire in the subset.

In [9]:
explainer = shap.TreeExplainer(model)
sv = explainer(X)

shap.plots.beeswarm(sv, max_display=15, show=False)
plt.gcf().savefig(FIG_DIR / "shap_beeswarm_l1.png", dpi=150, bbox_inches="tight")
plt.close("all")

shap.plots.bar(sv, max_display=15, show=False)
plt.gcf().savefig(FIG_DIR / "shap_bar_l1.png", dpi=150, bbox_inches="tight")
plt.close("all")

mean_abs = np.abs(sv.values).mean(0)
top = [feature_cols[i] for i in np.argsort(mean_abs)[::-1][:3]]
print("top features by mean |SHAP|:", top)

shap.plots.scatter(sv[:, top[0]], show=False)
plt.gcf().savefig(FIG_DIR / f"shap_dependence_{top[0]}.png", dpi=150, bbox_inches="tight")
plt.close("all")

idx = int(np.argmax((y.values == 1) * df[TARGET_COL].values))
shap.plots.waterfall(sv[idx], max_display=12, show=False)
plt.gcf().savefig(FIG_DIR / "shap_waterfall_l1.png", dpi=150, bbox_inches="tight")
plt.close("all")
print("largest mega-fire used for waterfall:", float(df[TARGET_COL].values[idx]), "ha")
print("saved SHAP figures to", FIG_DIR)

top features by mean |SHAP|: ['e', 'stl2', 'wind_speed']
largest mega-fire used for waterfall: 159812.58 ha
saved SHAP figures to /home/imorales/Desktop/XAI/XAI-project/latex/images


## 7. Environment (hardware / software)

In [10]:
def _cpu_model():
    """Devuelve el modelo de CPU leído de /proc/cpuinfo (fallback a platform.processor)."""
    try:
        for line in Path("/proc/cpuinfo").read_text().splitlines():
            if line.startswith("model name"):
                return line.split(":", 1)[1].strip()
    except Exception:
        pass
    return platform.processor() or "unknown"

def _ram_gb():
    """Devuelve la RAM total en GB leída de /proc/meminfo (None si falla)."""
    try:
        for line in Path("/proc/meminfo").read_text().splitlines():
            if line.startswith("MemTotal"):
                return round(int(line.split()[1]) / 1024 / 1024, 1)
    except Exception:
        return None

env = {
    "python": sys.version.split()[0],
    "os": platform.platform(),
    "cpu": _cpu_model(),
    "ram_gb": _ram_gb(),
    "xgboost": xgb.__version__,
    "shap": shap.__version__,
    "scikit_learn": sklearn.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "train_seconds": round(train_seconds, 2),
}
for k, v in env.items():
    print(f"{k:14s}: {v}")

python        : 3.12.13
os            : Linux-6.19.14-200.fc43.x86_64-x86_64-with-glibc2.42
cpu           : 11th Gen Intel(R) Core(TM) i7-1165G7 @ 2.80GHz
ram_gb        : 15.3
xgboost       : 3.2.0
shap          : 0.52.0
scikit_learn  : 1.9.0
numpy         : 2.4.6
pandas        : 3.0.3
train_seconds : 1.54


## 8. Persist model and metadata

In [11]:
model_path = DATA_MODELS / "xgboost_baseline_l1.json"
meta_path = DATA_MODELS / "xgboost_baseline_l1_meta.json"
model.save_model(str(model_path))

meta = {
    "label": "L1",
    "threshold_ha": MEGAFIRE_HA_THRESHOLD,
    "focus_regions": FOCUS_REGIONS,
    "n_events": int(len(X)),
    "n_positives": n_pos,
    "positive_rate": round(n_pos / len(X), 4),
    "n_features": len(feature_cols),
    "features": feature_cols,
    "cv": {
        "n_splits": N_SPLITS,
        "roc_auc_mean": round(float(roc_scores.mean()), 4),
        "roc_auc_std": round(float(roc_scores.std()), 4),
        "pr_auc_mean": round(float(pr_scores.mean()), 4),
        "pr_auc_std": round(float(pr_scores.std()), 4),
    },
    "scale_pos_weight": round(float(spw_global), 1),
    "xgb_params": XGB_PARAMS,
    "top_features": top,
    "environment": env,
}
meta_path.write_text(json.dumps(meta, indent=2, ensure_ascii=False))
print("saved:", model_path)
print("saved:", meta_path)

saved: /home/imorales/Desktop/XAI/XAI-project/data/models/xgboost_baseline_l1.json
saved: /home/imorales/Desktop/XAI/XAI-project/data/models/xgboost_baseline_l1_meta.json
